# 🔪 RingCutter — GNN Training
## Notebook 3: Train Graph Neural Network on Google Colab

**Instructions:**
1. Open this notebook in Google Colab
2. Runtime → Change runtime type → GPU
3. Upload your CSV files from `data/` folder
4. Run all cells
5. Download the trained model

### Step 1: Install Dependencies (Colab Only)

In [ ]:
# Run this cell ONLY on Google Colab
# Skip if running locally

# Uncomment the lines below if on Colab:
# !pip install torch-geometric
# !pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
# !pip install networkx faker scikit-learn

print('Dependencies ready ✅')

### Step 2: Upload Data (Colab Only)

In [ ]:
# Run this cell ONLY on Google Colab to upload files

# Uncomment if on Colab:
# from google.colab import files
# print('Upload synthetic_accounts.csv:')
# uploaded = files.upload()
# print('Upload synthetic_transactions.csv:')
# uploaded = files.upload()

import os
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# If running locally, set paths
ACCOUNTS_PATH = '../data/synthetic_accounts.csv'
TRANSACTIONS_PATH = '../data/synthetic_transactions.csv'

# If on Colab, use these paths instead:
# ACCOUNTS_PATH = 'synthetic_accounts.csv'
# TRANSACTIONS_PATH = 'synthetic_transactions.csv'

print('Data paths set ✅')

### Step 3: Import Libraries

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Try importing PyG
try:
    from torch_geometric.nn import SAGEConv, GATConv
    from torch_geometric.data import Data
    PYG = True
    print('PyTorch Geometric loaded ✅')
except:
    PYG = False
    print('PyTorch Geometric NOT available ⚠️')
    print('Will use fallback model')

### Step 4: Build Graph

In [ ]:
# Load data
accounts_df = pd.read_csv(ACCOUNTS_PATH)
txns_df = pd.read_csv(TRANSACTIONS_PATH)
txns_df['timestamp'] = pd.to_datetime(txns_df['timestamp'])

print(f'Accounts: {len(accounts_df)}')
print(f'Transactions: {len(txns_df)}')

# Build graph
graph = nx.MultiDiGraph()

# Add account nodes
for _, row in accounts_df.iterrows():
    graph.add_node(row['account_id'], node_type='ACCOUNT',
                   account_age_days=row['account_age_days'],
                   avg_monthly_balance=row['avg_monthly_balance'],
                   is_mule=row['is_mule'],
                   mule_ring_id=row['mule_ring_id'])

# Add transaction edges
for _, txn in txns_df.iterrows():
    graph.add_edge(txn['from_account'], txn['to_account'],
                   edge_type='TRANSFERS_TO',
                   amount=txn['amount'],
                   channel=txn['channel'],
                   is_fraud=txn['is_fraud'])

# Add device sharing edges
device_to_accounts = defaultdict(list)
for _, row in accounts_df.iterrows():
    device_to_accounts[row['primary_device_id']].append(row['account_id'])

shared_count = 0
for dev_id, acc_list in device_to_accounts.items():
    if len(acc_list) > 1:
        for i in range(len(acc_list)):
            for j in range(i + 1, len(acc_list)):
                graph.add_edge(acc_list[i], acc_list[j], edge_type='SHARES_DEVICE')
                shared_count += 1

print(f'Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges')
print(f'Shared device edges: {shared_count}')

### Step 5: Compute Node Features

In [ ]:
# Compute features for each account
print('Computing node features...')

account_nodes = [n for n, d in graph.nodes(data=True) if d.get('node_type') == 'ACCOUNT']
node_to_idx = {node: idx for idx, node in enumerate(account_nodes)}

feature_names = ['account_age_days', 'avg_monthly_balance',
                 'total_txn_count', 'total_amount_sent',
                 'total_amount_received', 'unique_counterparties',
                 'channel_diversity', 'avg_txn_amount',
                 'max_txn_amount', 'txn_frequency_per_day',
                 'night_txn_ratio', 'cross_channel_ratio']

features = []
labels = []

for node in account_nodes:
    nd = graph.nodes[node]
    sent = txns_df[txns_df['from_account'] == node]
    received = txns_df[txns_df['to_account'] == node]
    all_txns = pd.concat([sent, received])

    feat = [
        nd.get('account_age_days', 0),
        nd.get('avg_monthly_balance', 0),
        len(all_txns),
        sent['amount'].sum() if len(sent) > 0 else 0,
        received['amount'].sum() if len(received) > 0 else 0,
        len(set(sent['to_account'].tolist() + received['from_account'].tolist())),
        all_txns['channel'].nunique() if len(all_txns) > 0 else 0,
        all_txns['amount'].mean() if len(all_txns) > 0 else 0,
        all_txns['amount'].max() if len(all_txns) > 0 else 0,
        len(all_txns) / max(nd.get('account_age_days', 1), 1),
        len(all_txns[all_txns['hour'].isin([23,0,1,2,3,4,5])]) / max(len(all_txns), 1),
        all_txns['channel'].nunique() / max(len(all_txns), 1) if len(all_txns) > 0 else 0
    ]
    features.append(feat)
    labels.append(nd.get('is_mule', 0))

features = np.array(features, dtype=np.float32)
scaler = StandardScaler()
features = scaler.fit_transform(features)

print(f'Features shape: {features.shape}')
print(f'Mule accounts: {sum(labels)}')
print(f'Normal accounts: {len(labels) - sum(labels)}')

### Step 6: Prepare PyG Data

In [ ]:
# Convert to tensors
x = torch.FloatTensor(features)
y = torch.FloatTensor(labels)

# Build edge index
edge_list = []
for u, v, d in graph.edges(data=True):
    if d.get('edge_type') == 'TRANSFERS_TO' and u in node_to_idx and v in node_to_idx:
        edge_list.append([node_to_idx[u], node_to_idx[v]])
    elif d.get('edge_type') == 'SHARES_DEVICE' and u in node_to_idx and v in node_to_idx:
        edge_list.append([node_to_idx[u], node_to_idx[v]])
        edge_list.append([node_to_idx[v], node_to_idx[u]])

edge_index = torch.LongTensor(edge_list).t().contiguous()

# Train/test split
num_nodes = len(account_nodes)
indices = np.arange(num_nodes)
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[train_idx] = True
test_mask[test_idx] = True

print(f'Nodes: {num_nodes}')
print(f'Edges: {edge_index.shape[1]}')
print(f'Features: {x.shape[1]}')
print(f'Train: {train_mask.sum().item()}')
print(f'Test: {test_mask.sum().item()}')

### Step 7: Define GNN Model

In [ ]:
if PYG:
    class RingCutterGNN(torch.nn.Module):
        def __init__(self, num_features, hidden_dim=64):
            super(RingCutterGNN, self).__init__()
            self.conv1 = SAGEConv(num_features, hidden_dim)
            self.conv2 = SAGEConv(hidden_dim, hidden_dim // 2)
            self.classifier = torch.nn.Linear(hidden_dim // 2, 1)
            self.dropout = torch.nn.Dropout(0.3)

        def forward(self, x, edge_index):
            h = self.conv1(x, edge_index)
            h = F.relu(h)
            h = self.dropout(h)
            h = self.conv2(h, edge_index)
            h = F.relu(h)
            h = self.dropout(h)
            out = self.classifier(h)
            out = torch.sigmoid(out)
            return out.squeeze()

    model = RingCutterGNN(num_features=x.shape[1], hidden_dim=64)
    print(f'GNN Model created ✅')
    print(model)
else:
    class FallbackModel(torch.nn.Module):
        def __init__(self, num_features, hidden_dim=64):
            super(FallbackModel, self).__init__()
            self.layer1 = torch.nn.Linear(num_features, hidden_dim)
            self.layer2 = torch.nn.Linear(hidden_dim, hidden_dim // 2)
            self.layer3 = torch.nn.Linear(hidden_dim // 2, 1)
            self.dropout = torch.nn.Dropout(0.3)

        def forward(self, x, edge_index=None):
            h = F.relu(self.layer1(x))
            h = self.dropout(h)
            h = F.relu(self.layer2(h))
            h = self.dropout(h)
            out = torch.sigmoid(self.layer3(h))
            return out.squeeze()

    model = FallbackModel(num_features=x.shape[1], hidden_dim=64)
    print(f'Fallback Model created ⚠️')
    print(model)

### Step 8: Train the Model

In [ ]:
# Training setup
num_normal = int((y == 0).sum())
num_mule = int((y == 1).sum())
pos_weight = torch.FloatTensor([num_normal / max(num_mule, 1)])

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# Training loop
epochs = 200
train_losses = []
test_f1s = []
best_f1 = 0
best_state = None

print('Training started...')
print(f'{"Epoch":>6} | {"Loss":>8} | {"F1":>8} | {"Prec":>8} | {"Rec":>8}')
print('-' * 45)

model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    out = model(x, edge_index)
    loss = criterion(out[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    if (epoch + 1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            pred = model(x, edge_index)
            pred_labels = (pred > 0.5).float()
            test_pred = pred_labels[test_mask].numpy()
            test_true = y[test_mask].numpy()
            f1 = f1_score(test_true, test_pred, zero_division=0)
            prec = precision_score(test_true, test_pred, zero_division=0)
            rec = recall_score(test_true, test_pred, zero_division=0)
            test_f1s.append(f1)
            print(f'{epoch+1:>6} | {loss.item():>8.4f} | {f1:>8.4f} | {prec:>8.4f} | {rec:>8.4f}')
            if f1 > best_f1:
                best_f1 = f1
                best_state = model.state_dict().copy()
        model.train()

print(f'\nBest F1: {best_f1:.4f}')

### Step 9: Evaluate & Visualize Results

In [ ]:
# Load best model
if best_state is not None:
    model.load_state_dict(best_state)

# Final evaluation
model.eval()
with torch.no_grad():
    pred = model(x, edge_index)
    pred_labels = (pred > 0.5).float()
    test_pred = pred_labels[test_mask].numpy()
    test_true = y[test_mask].numpy()

print('\n' + '=' * 60)
print('  FINAL TEST SET RESULTS')
print('=' * 60)
print(classification_report(test_true, test_pred, target_names=['Normal', 'Mule']))

# Confusion Matrix
cm = confusion_matrix(test_true, test_pred)
print(f'Confusion Matrix:')
print(f'  TN={cm[0][0]:4d}  FP={cm[0][1]:4d}')
print(f'  FN={cm[1][0]:4d}  TP={cm[1][1]:4d}')

In [ ]:
# Plot training loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, color='#FF4B4B')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(range(20, epochs + 1, 20), test_f1s, color='#00CC66', marker='o')
axes[1].set_title('Test F1 Score')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')

plt.tight_layout()
plt.savefig('../data/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved ✅')

In [ ]:
# Score distribution
with torch.no_grad():
    all_scores = model(x, edge_index).numpy()

normal_scores = all_scores[y.numpy() == 0]
mule_scores = all_scores[y.numpy() == 1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(normal_scores, bins=50, color='#00CC66', alpha=0.7, label=f'Normal (n={len(normal_scores)})')
ax.hist(mule_scores, bins=50, color='#FF4B4B', alpha=0.7, label=f'Mule (n={len(mule_scores)})')
ax.axvline(x=0.5, color='white', linestyle='--', label='Threshold (0.5)')
ax.set_title('GNN Score Distribution: Normal vs Mule')
ax.set_xlabel('Mule Risk Score')
ax.set_ylabel('Count')
ax.legend()
plt.savefig('../data/score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Score distribution saved ✅')

### Step 10: Save Model

In [ ]:
# Save the trained model
import os
os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), '../models/ringcutter_gnn.pth')
print('Model saved to models/ringcutter_gnn.pth ✅')

# If on Colab, download the model file:
# from google.colab import files
# files.download('../models/ringcutter_gnn.pth')

print(f'\nModel Summary:')
print(f'  Architecture: {"GraphSAGE GNN" if PYG else "Fallback NN"}')
print(f'  Input features: {x.shape[1]}')
print(f'  Best F1 Score: {best_f1:.4f}')
print(f'  Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'\n✅ Training complete! Model ready for deployment.')